# Step 2: Federated Training with Gradient Projection (no Flower)

In [1]:
import sys
import subprocess

print("=" * 60)
print("GPU / ENVIRONMENT CHECK")
print("=" * 60)
print(f"Python: {sys.version}")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        n_gpus = torch.cuda.device_count()
        print(f"Number of GPUs: {n_gpus}")
        for i in range(n_gpus):
            props = torch.cuda.get_device_properties(i)
            mem_gb = props.total_memory / 1024 ** 3
            print(f"  GPU {i}: {props.name} ({mem_gb:.1f} GB)")
    else:
        print("WARNING: No GPU detected. Training will be very slow on CPU.")
except ImportError:
    print("PyTorch not yet installed — will be installed in next cell.")
print("=" * 60)

GPU / ENVIRONMENT CHECK
Python: 3.12.11 | packaged by conda-forge | (main, Jun  4 2025, 14:45:31) [GCC 13.3.0]
PyTorch: 2.8.0+cu128
CUDA available: True
CUDA version: 12.8
Number of GPUs: 2
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition (95.0 GB)
  GPU 1: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition (95.0 GB)


In [2]:
%pip install -q transformers datasets peft

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import warnings
from pathlib import Path
from typing import Dict, List

import numpy as np
import torch
from torch.utils.data import Dataset

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

warnings.filterwarnings("ignore")
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

MODEL_NAME = "Qwen/Qwen2.5-7B"

LORA_CONFIG = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)

N_CLIENTS   = 10
N_ROUNDS    = 10   # ← changed from 3 to 10
BATCH_SIZE  = 2
MAX_SEQ_LEN = 512
DTYPE       = torch.bfloat16

BASE_DIR    = Path.home() / "federated-ethical-llm"
MODELS_DIR  = BASE_DIR / "models"
DATA_DIR    = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"

for _d in [MODELS_DIR, DATA_DIR, RESULTS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

SAFETY_SUBSPACE_PATH = MODELS_DIR / "safety_subspace.pt"
SAFETY_LORA_PATH     = MODELS_DIR / "safety_lora"

# Save with round count in filename — keeps 3-round results intact
FEDAVG_LORA_PATH = MODELS_DIR / f"fedavg_lora_{N_ROUNDS}r.pt"
STEP2_DONE_FLAG  = MODELS_DIR / f"step2_{N_ROUNDS}r_done.flag"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"N_ROUNDS    : {N_ROUNDS}")
print(f"Output path : {FEDAVG_LORA_PATH}")
print(f"Device      : {DEVICE}")

2026-03-19 18:57:01.179006: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-19 18:57:01.189739: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773964621.202628    9544 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773964621.206803    9544 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773964621.217098    9544 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

N_ROUNDS    : 10
Output path : /home/jovyan/federated-ethical-llm/models/fedavg_lora_10r.pt
Device      : cuda


In [4]:
# Shared helpers (same as Step 1 — reproduced for standalone execution)

def load_base_model_and_tokenizer(model_name: str = MODEL_NAME, for_training: bool = True):
    """Load Qwen2.5-7B in bfloat16 with device_map={\"\"\: 0} for training."""
    print(f"Loading tokenizer for {model_name} ...")
    tokenizer = AutoTokenizer.from_pretrained(
        model_name, trust_remote_code=True, padding_side="right"
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    device_map = {"" : 0} if for_training else "auto"
    print(f"Loading model (bfloat16, device_map={device_map}) ...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=DTYPE,
        device_map=device_map,
        trust_remote_code=True,
    )
    model.config.use_cache = False
    model.enable_input_require_grads()
    print("Model loaded.")
    return model, tokenizer


def apply_lora(model, lora_config: LoraConfig = LORA_CONFIG):
    """Wrap model with PEFT LoRA adapter."""
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model


class TextDataset(Dataset):
    """Simple tokenised text dataset with -100 label masking for padding."""

    def __init__(self, texts: List[str], tokenizer, max_length: int = MAX_SEQ_LEN):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_tensors="pt",
        )

    def __len__(self):
        return self.encodings["input_ids"].shape[0]

    def __getitem__(self, idx):
        input_ids      = self.encodings["input_ids"][idx]
        attention_mask = self.encodings["attention_mask"][idx]
        labels         = input_ids.clone()
        labels[attention_mask == 0] = -100   # mask padding tokens
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


class SafetySubspace:
    """Container for per-layer safety basis matrices (U tensors)."""

    def __init__(self):
        self.bases: Dict[str, torch.Tensor] = {}

    @classmethod
    def load(cls, path: Path = SAFETY_SUBSPACE_PATH):
        obj = cls()
        obj.bases = torch.load(str(path), map_location="cpu")
        print(f"Safety subspace loaded from {path} ({len(obj.bases)} layers)")
        return obj


print("Shared helpers defined.")

Shared helpers defined.


In [5]:
# LoRA weight utilities — keep ONE model in GPU memory, only swap small dicts

def get_lora_state(model) -> dict:
    """Extract LoRA weights as CPU float32 dict (tiny ~160MB)."""
    return {
        n: p.detach().cpu().float().clone()
        for n, p in model.named_parameters()
        if "lora_" in n
    }


def set_lora_state(model, state: dict):
    """Load LoRA weights back into model in-place."""
    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in state:
                p.copy_(state[n].to(p.device, dtype=p.dtype))


def average_lora_states(states: list) -> dict:
    """FedAvg: element-wise mean of LoRA weight dicts."""
    avg = {}
    for key in states[0]:
        avg[key] = torch.stack([s[key] for s in states]).mean(dim=0)
    return avg


print("LoRA weight utilities defined.")

LoRA weight utilities defined.


In [6]:
# Gradient projection hook + client training function

def make_projection_hook(U: torch.Tensor):
    """Project gradient away from safety subspace: grad = grad - U @ U.T @ grad"""
    def hook(grad):
        if grad is None:
            return grad
        Ud = U.to(grad.device, dtype=grad.dtype)
        return grad - Ud @ (Ud.T @ grad)
    return hook


def register_safety_hooks(model, safety_bases: dict) -> list:
    """Attach projection hooks to all lora_B params. Returns handle list."""
    handles = []
    for name, param in model.named_parameters():
        if "lora_B" not in name:
            continue
        layer_key = name.replace(".lora_B.default", "").replace(".lora_B", "")
        if layer_key in safety_bases:
            h = param.register_hook(make_projection_hook(safety_bases[layer_key]))
            handles.append(h)
    return handles


def remove_hooks(handles: list):
    for h in handles:
        h.remove()


def train_client(model, dataset, tokenizer, lr: float = 2e-4, epochs: int = 1) -> float:
    """Train model on dataset for one epoch. Returns avg loss."""
    from torch.utils.data import DataLoader

    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr
    )
    model.train()
    total_loss, n = 0.0, 0
    for batch in loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        optimizer.zero_grad()
        outputs.loss.backward()
        optimizer.step()
        total_loss += outputs.loss.item()
        n += 1
    return total_loss / max(n, 1)


print("Projection hook and train_client defined.")

Projection hook and train_client defined.


In [7]:
# prepare_client_datasets — split rejected_texts into N_CLIENTS shards

def prepare_client_datasets(
    rejected_texts: List[str],
    tokenizer,
    n_clients: int = N_CLIENTS,
    seed: int = 42,
) -> List[TextDataset]:
    """Split rejected responses into n_clients shards; return a TextDataset per client."""
    rng     = np.random.default_rng(seed)
    indices = rng.permutation(len(rejected_texts))
    shards  = np.array_split(indices, n_clients)

    client_datasets = []
    for i, shard_idx in enumerate(shards):
        texts = [rejected_texts[j] for j in shard_idx]
        ds    = TextDataset(texts, tokenizer)
        client_datasets.append(ds)
        print(f"  Client {i:2d}: {len(texts)} samples")

    return client_datasets


print("prepare_client_datasets defined.")

prepare_client_datasets defined.


In [8]:
# Main federated simulation loop (no Flower)

def run_federated_simulation(
    model,
    tokenizer,
    client_datasets,
    safety_bases: dict,
    n_rounds: int = N_ROUNDS,
):
    """
    Federated simulation keeping ONE model in memory.
    Per round: for each client, set LoRA weights, train with constraint, collect weights.
    Then FedAvg aggregate and update global model.
    """
    # Save initial global LoRA state
    global_state = get_lora_state(model)
    round_losses  = []

    for round_num in range(n_rounds):
        print(f"\n{'=' * 50}")
        print(f"Round {round_num + 1}/{n_rounds}")
        print(f"{'=' * 50}")
        client_states = []
        client_losses = []

        for cid, dataset in enumerate(client_datasets):
            # Restore global weights for this client
            set_lora_state(model, global_state)

            # Register gradient projection hooks
            handles = register_safety_hooks(model, safety_bases)

            # Train locally on biased data
            loss = train_client(model, dataset, tokenizer)

            # Remove hooks and collect updated LoRA state
            remove_hooks(handles)
            client_states.append(get_lora_state(model))
            client_losses.append(loss)
            print(f"  Client {cid:2d}: loss={loss:.4f}")

        # FedAvg aggregation
        global_state = average_lora_states(client_states)
        avg_loss     = sum(client_losses) / len(client_losses)
        round_losses.append(avg_loss)
        print(f"Round {round_num + 1} avg loss: {avg_loss:.4f}")
        torch.cuda.empty_cache()

    # Apply final global state to model
    set_lora_state(model, global_state)
    return model, round_losses, global_state


print("run_federated_simulation defined.")

run_federated_simulation defined.


In [9]:
# Run cell

if STEP2_DONE_FLAG.exists():
    print(f"Step 2 ({N_ROUNDS} rounds) already complete — skipping.")
else:
    print("Loading safety subspace ...")
    safety_subspace_obj = SafetySubspace.load(SAFETY_SUBSPACE_PATH)
    safety_bases = safety_subspace_obj.bases

    print("Loading rejected texts ...")
    rejected_texts: List[str] = torch.load(str(DATA_DIR / "rejected_texts.pt"))

    print("Loading model ...")
    model, tokenizer = load_base_model_and_tokenizer(for_training=True)

    if SAFETY_LORA_PATH.exists():
        print(f"Loading LoRA from {SAFETY_LORA_PATH} ...")
        model = PeftModel.from_pretrained(model, str(SAFETY_LORA_PATH), is_trainable=True)
    else:
        model = apply_lora(model)

    print(f"\nPreparing {N_CLIENTS} client datasets ...")
    client_datasets = prepare_client_datasets(rejected_texts, tokenizer)

    print(f"\nStarting federated simulation ({N_ROUNDS} rounds) ...")
    model, round_losses, global_state = run_federated_simulation(
        model, tokenizer, client_datasets, safety_bases, n_rounds=N_ROUNDS)

    # Save with round count in filename
    torch.save(global_state, str(FEDAVG_LORA_PATH))
    torch.save(round_losses, str(RESULTS_DIR / f"round_losses_{N_ROUNDS}r.pt"))

    STEP2_DONE_FLAG.touch()

    print("\n" + "=" * 50)
    print(f"STEP 2 COMPLETE — {N_ROUNDS} ROUNDS")
    print("=" * 50)
    for i, loss in enumerate(round_losses):
        print(f"  Round {i+1:2d}: avg loss = {loss:.4f}")
    print(f"\n  Saved: {FEDAVG_LORA_PATH}")

    del model
    torch.cuda.empty_cache()

Loading safety subspace ...
Safety subspace loaded from /home/jovyan/federated-ethical-llm/models/safety_subspace.pt (196 layers)
Loading rejected texts ...
Loading model ...
Loading tokenizer for Qwen/Qwen2.5-7B ...
Loading model (bfloat16, device_map={'': 0}) ...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded.
Loading LoRA from /home/jovyan/federated-ethical-llm/models/safety_lora ...

Preparing 10 client datasets ...
  Client  0: 100 samples
  Client  1: 100 samples
  Client  2: 100 samples
  Client  3: 100 samples
  Client  4: 100 samples
  Client  5: 100 samples
  Client  6: 100 samples
  Client  7: 100 samples
  Client  8: 100 samples
  Client  9: 100 samples

Starting federated simulation (10 rounds) ...

Round 1/10
  Client  0: loss=1.8309
  Client  1: loss=1.7936
  Client  2: loss=1.8623
  Client  3: loss=1.8117
  Client  4: loss=1.7775
  Client  5: loss=1.8125
  Client  6: loss=1.8136
  Client  7: loss=1.8094
  Client  8: loss=1.8353
  Client  9: loss=1.7889
Round 1 avg loss: 1.8136

Round 2/10
  Client  0: loss=1.7814
  Client  1: loss=1.7367
  Client  2: loss=1.8216
  Client  3: loss=1.7525
  Client  4: loss=1.7327
  Client  5: loss=1.7295
  Client  6: loss=1.7872
  Client  7: loss=1.7613
  Client  8: loss=1.7919
  Client  9: loss=1.7475
Round 2 avg loss: 1.7642

Roun

In [ ]:
# Run no-projection federated baseline (biased clients, NO safety constraint)
# Uses GPU 1 to avoid OOM when GPU 0 is occupied by other users.
# Starts from a fresh LoRA (no safety_lora) — keeps everything on GPU 1.
FEDAVG_NOPROJ_PATH = MODELS_DIR / f"fedavg_noproj_{N_ROUNDS}r.pt"
STEP2B_DONE_FLAG   = MODELS_DIR / f"step2b_{N_ROUNDS}r_done.flag"

if STEP2B_DONE_FLAG.exists():
    print(f"No-projection baseline already done — skipping.")
    print(f"  Saved at: {FEDAVG_NOPROJ_PATH}")
else:
    DEVICE_NP = torch.device("cuda:1" if torch.cuda.device_count() > 1 else "cuda:0")
    gpu_id = 1 if torch.cuda.device_count() > 1 else 0
    print(f"Using device: {DEVICE_NP}")

    # Load base model directly onto GPU 1
    from transformers import AutoTokenizer, AutoModelForCausalLM
    print("Loading tokenizer ...")
    tok_np = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, padding_side="right")
    if tok_np.pad_token is None:
        tok_np.pad_token = tok_np.eos_token

    print(f"Loading model onto GPU {gpu_id} ...")
    base_np = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=DTYPE,
        device_map={"": gpu_id},
        trust_remote_code=True,
    )
    base_np.config.use_cache = False
    base_np.enable_input_require_grads()

    # Apply fresh LoRA (no PeftModel.from_pretrained — avoids loading onto GPU 0)
    model_np = get_peft_model(base_np, LORA_CONFIG)
    model_np.print_trainable_parameters()
    print("Model ready on GPU 1.")

    rejected_texts_np = torch.load(str(DATA_DIR / "rejected_texts.pt"))
    client_ds_np = prepare_client_datasets(rejected_texts_np, tok_np)

    # Inline federated loop on DEVICE_NP — no projection hooks
    global_state_np = get_lora_state(model_np)
    round_losses_np = []

    for round_num in range(N_ROUNDS):
        print(f"\n{'='*50}\nRound {round_num+1}/{N_ROUNDS}\n{'='*50}")
        client_states, client_losses = [], []

        for cid, dataset in enumerate(client_ds_np):
            set_lora_state(model_np, global_state_np)
            # NO safety hooks — pure biased training
            from torch.utils.data import DataLoader
            loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
            optimizer = torch.optim.AdamW(
                [p for p in model_np.parameters() if p.requires_grad], lr=2e-4)
            model_np.train()
            total_loss, n = 0.0, 0
            for batch in loader:
                ids  = batch["input_ids"].to(DEVICE_NP)
                mask = batch["attention_mask"].to(DEVICE_NP)
                lbls = batch["labels"].to(DEVICE_NP)
                out  = model_np(input_ids=ids, attention_mask=mask, labels=lbls)
                optimizer.zero_grad(); out.loss.backward(); optimizer.step()
                total_loss += out.loss.item(); n += 1
            loss = total_loss / max(n, 1)
            client_states.append(get_lora_state(model_np))
            client_losses.append(loss)
            print(f"  Client {cid:2d}: loss={loss:.4f}")

        global_state_np = average_lora_states(client_states)
        avg = sum(client_losses) / len(client_losses)
        round_losses_np.append(avg)
        print(f"Round {round_num+1} avg loss: {avg:.4f}")
        torch.cuda.empty_cache()

    torch.save(global_state_np, str(FEDAVG_NOPROJ_PATH))
    torch.save(round_losses_np, str(RESULTS_DIR / f"round_losses_noproj_{N_ROUNDS}r.pt"))
    STEP2B_DONE_FLAG.touch()

    print(f"\nNo-projection baseline complete.")
    for idx, loss in enumerate(round_losses_np):
        print(f"  Round {idx+1:2d}: avg loss = {loss:.4f}")
    print(f"Saved: {FEDAVG_NOPROJ_PATH}")

    del model_np, base_np
    torch.cuda.empty_cache()